[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20collection/wikipedia_election_scraper.ipynb)

# Scraping Wikipedia: the 2024 general election, party by party, state by state

**DA2402 · Data Curation and Visualization · Dr. Arun B Ayyar**

The question we want answered, as one tidy table: **which party contested how many seats in which
state, and how many did it win?**

The source is the Wikipedia article
[2024 Indian general election](https://en.wikipedia.org/wiki/2024_Indian_general_election).
On the data-collection ladder this is rung 3 — parse the server's HTML — but it is the *easiest*
version of rung 3 you will ever meet: Wikipedia is static, server-rendered HTML, full of real
`<table>` elements, and `pandas.read_html()` parses every one of them in a single call. No
Selenium, no hidden API, no session dance.

That does not mean no work. The page has **72 tables**, the ones we want carry footnote markers,
hidden "Total" rows and merged cells, and next month an editor may reorder everything. Most of
this notebook is the part that always survives contact with real data: *finding* the right tables
and *cleaning* them.

## 1 · Fetch the page once, politely

`pd.read_html(url)` would work, but fetching with `requests` first is better manners and better
engineering: we send an honest User-Agent that says who we are (Wikipedia asks for this), and we
hit the server once, keeping the HTML in memory while we experiment on it.

In [1]:
import requests
import pandas as pd
from io import StringIO

URL = "https://en.wikipedia.org/wiki/2024_Indian_general_election"
UA = {"User-Agent": "DA2402-course-notebook/1.0 (IIT Madras BS DSAI; educational use)"}

html = requests.get(URL, headers=UA, timeout=30).text
print(f"{len(html):,} characters of HTML")

2,060,189 characters of HTML


## 2 · One call, every table

`read_html` walks the whole document and returns a list of DataFrames, one per `<table>`.

In [2]:
tables = pd.read_html(StringIO(html))
print(f"{len(tables)} tables on the page")

72 tables on the page


## 3 · Find our tables by *signature*, not by position

Table 5 today may be table 9 after the next edit — a Wikipedia article is a living document, so
**never hard-code a table index**. Instead we describe the table we want: the party-wise results
tables are the ones with a `Seats contested` column. Four tables match: the NDA table, **two**
INDIA-bloc tables — the article splits seats contested *under* the alliance's seat-sharing pact
from seats its member parties contested *outside* it — and an "other notable parties" table,
which has no `Seats won` column at all. *The absence of a column is data too*, and we will have
to think about what it means before we are done.

In [3]:
def colnames(t):
    """Column labels as flat lowercase strings (works for MultiIndex headers too)."""
    if isinstance(t.columns, pd.MultiIndex):
        return [" ".join(str(x) for x in c).lower() for c in t.columns]
    return [str(c).lower() for c in t.columns]

matches = [i for i, t in enumerate(tables)
           if any("seats contested" in c for c in colnames(t))]
for i in matches:
    has_won = any("seats won" in c for c in colnames(tables[i]))
    print(f"table {i}: {tables[i].shape}, has a 'Seats won' column: {has_won}")

table 5: (60, 7), has a 'Seats won' column: True
table 6: (76, 7), has a 'Seats won' column: True
table 7: (60, 7), has a 'Seats won' column: True
table 8: (83, 5), has a 'Seats won' column: False


## 4 · Look at the raw material before touching it

Always. The first matching table shows everything we will have to clean:

- an unnamed first column (it holds the party's colour swatch — no text, so it parses as `NaN`);
- footnote markers riding on the numbers (`75[58]` is a string, not a number);
- two `Seats contested` columns — the per-state figure and the party's national total, which
  `read_html` repeats on every row because the original cell is merged (`rowspan`);
- and, invisible here but waiting at the bottom, a **`Total` row for the whole alliance**. Sum a
  column without dropping it and every seat is counted twice.

In [4]:
raw = tables[matches[0]]
raw.head(4)

,Party,Party.1,State/UT,Seats contested,Seats contested.1,Seats won,Seats won.1
0,NaN,Bharatiya Janata Party,Uttar Pradesh,75[58],441[59],33,240
1,NaN,Bharatiya Janata Party,West Bengal,42,441[59],12,240
2,NaN,Bharatiya Janata Party,Madhya Pradesh,29,441[59],29,240
3,NaN,Bharatiya Janata Party,Maharashtra,28,441[59],9,240


In [5]:
raw.tail(2)  # the alliance "Total" row that would double every sum

,Party,Party.1,State/UT,Seats contested,Seats contested.1,Seats won,Seats won.1
58,NaN,Independent,Tamil Nadu,1,1,0,0
59,Total,Total,Total,541,541,293,293


## 5 · Clean one table — then the same function cleans them all

Cleaning steps, each one earned by something we just saw: name the columns; drop the swatch;
drop the `Total` row (we keep its number aside — it becomes a free check on our work); strip
`[58]`-style footnotes; convert to integers. The alliance label comes from the table's own
content, not its position — the table containing the BJP is the NDA table wherever it appears,
and both tables containing the INC are INDIA-bloc tables. Because one alliance spans two tables,
the kept-aside totals are *added up* per alliance, not overwritten.

In [6]:
import re

def clean(t):
    """Tidy one alliance table -> (per-state rows, wikipedia's own total-won figure)."""
    t = t.copy()
    t.columns = ["swatch", "party", "state", "contested", "contested_total", "won", "won_total"]
    strip = lambda s: re.sub(r"\[\d+\]", "", str(s))          # "75[58]" -> "75"
    for c in ["contested", "won", "won_total"]:
        t[c] = pd.to_numeric(t[c].map(strip), errors="coerce")
    total_row = t[t["party"].eq("Total")]
    wiki_total = int(total_row["won"].iloc[0]) if len(total_row) else None
    t = t[t["party"].ne("Total")].dropna(subset=["state"])
    return t[["party", "state", "contested", "won"]], wiki_total

def alliance_of(t):
    parties = set(t.iloc[:, 1].astype(str))
    if "Bharatiya Janata Party" in parties:   return "NDA"
    if "Indian National Congress" in parties: return "INDIA bloc"
    return "Unaligned"

frames, checks = [], {}
for i in matches:
    if not any("seats won" in c for c in colnames(tables[i])):
        continue                                   # the no-wins table — handled in step 6
    tidy, wiki_total = clean(tables[i])
    tidy.insert(0, "alliance", alliance_of(tables[i]))
    frames.append(tidy)
    label = tidy["alliance"].iloc[0]
    checks[label] = checks.get(label, 0) + wiki_total   # INDIA bloc spans two tables

results = pd.concat(frames, ignore_index=True)
print(f"{len(results)} party-state rows, {results['party'].nunique()} parties")
results.head(8)

193 party-state rows, 52 parties


,alliance,party,state,contested,won
0,NDA,Bharatiya Janata Party,Uttar Pradesh,75,33
1,NDA,Bharatiya Janata Party,West Bengal,42,12
2,NDA,Bharatiya Janata Party,Madhya Pradesh,29,29
3,NDA,Bharatiya Janata Party,Maharashtra,28,9
4,NDA,Bharatiya Janata Party,Gujarat,26,25
5,NDA,Bharatiya Janata Party,Karnataka,25,17
6,NDA,Bharatiya Janata Party,Rajasthan,25,14
7,NDA,Bharatiya Janata Party,Tamil Nadu,23,0


## 6 · The table with the missing column — and the trap in it

The fourth table, "other notable parties", records only seats *contested*. The tempting move is
to fill `won = 0` for all of them — resist it. The BSP really did contest 424 seats and win
none, but this same table also holds the YSR Congress Party (won 4 in Andhra Pradesh), the
AIMIM, the Shiromani Akali Dal and other parties that **did** win — the article simply never
breaks their wins down by state in this section. We record what the source actually says:
`won = <NA>`, *unknown at state level*. **A missing value is honest; a fabricated zero is a
bug** that would survive every later computation without ever looking wrong.

Two more quirks: the header is a two-level `MultiIndex` (flatten it first), and one "party" row
is really an aggregate — "Unrecognised parties" — which we drop along with the `Total` row.

In [7]:
nowins = next(tables[i] for i in matches
              if not any("seats won" in c for c in colnames(tables[i])))
nowins = nowins.copy()
nowins.columns = ["swatch", "party", "state", "contested", "contested_total"][:len(nowins.columns)]
nowins["contested"] = pd.to_numeric(
    nowins["contested"].map(lambda s: re.sub(r"\[\d+\]", "", str(s))), errors="coerce")
nowins = (nowins[~nowins["party"].isin(["Total", "Unrecognised parties"])]
          .dropna(subset=["party", "state", "contested"])
          [["party", "state", "contested"]])
nowins.insert(0, "alliance", "Other")
nowins["won"] = pd.NA          # unknown at state level — NOT zero

results = pd.concat([results, nowins], ignore_index=True)
results = results.astype({"contested": "Int64", "won": "Int64"})
print(f"{len(results)} rows, {results['party'].nunique()} parties, "
      f"{results['state'].nunique()} states/UTs")

259 rows, 78 parties, 40 states/UTs


## 7 · Check the work before believing it

Three checks, from known facts to internal consistency:

1. the BJP won 240 seats and the INC 99 — public record;
2. summing our per-state rows must reproduce the alliance totals Wikipedia itself printed in the
   `Total` rows we dropped;
3. the alliance rows should account for 527 of the 543 seats — NDA 293 + INDIA 234. The other
   16 went to independents (7 seats) and to parties from the no-`won`-column table, which is
   exactly the gap our `<NA>` values are marking.

One honest caveat found while building this: in the INDIA-bloc table the DMK and KMDK **share a
merged total cell** (KMDK contested on the DMK symbol), so those two parties' *national-total*
column disagrees with their per-state sum by one seat. Our per-state numbers are the correct
ones — which is exactly why this notebook sums states instead of trusting the totals column.

In [8]:
by_party = results.groupby("party")[["contested", "won"]].sum()
assert by_party.loc["Bharatiya Janata Party", "won"] == 240
assert by_party.loc["Indian National Congress", "won"] == 99

for alliance, wiki_total in checks.items():
    ours = results.loc[results["alliance"].eq(alliance), "won"].sum()
    flag = "ok" if ours == wiki_total else "MISMATCH"
    print(f"{alliance:11s} our sum {ours:3d}   wikipedia's Total row {wiki_total:3d}   {flag}")

alliance_seats = int(results["won"].sum())          # <NA> rows are skipped
print(f"\nalliance rows account for {alliance_seats} of 543 seats; the other "
      f"{543 - alliance_seats} belong to independents and to 'Other' parties, "
      f"whose per-state wins the article does not tabulate — our <NA> rows")

NDA         our sum 293   wikipedia's Total row 293   ok
INDIA bloc  our sum 234   wikipedia's Total row 234   ok

alliance rows account for 527 of 543 seats; the other 16 belong to independents and to 'Other' parties, whose per-state wins the article does not tabulate — our <NA> rows


## 8 · The answer

The tidy table *is* the answer to "which party contested how many seats in which state and won
how many" — every row one (party, state) pair. From it, any summary is a one-liner.

In [9]:
results.sort_values(["alliance", "party", "contested"],
                    ascending=[True, True, False], ignore_index=True)

,alliance,party,state,contested,won
0,INDIA bloc,Aam Aadmi Party,Punjab,13,3
1,INDIA bloc,Aam Aadmi Party,Delhi,4,0
2,INDIA bloc,Aam Aadmi Party,Gujarat,2,0
3,INDIA bloc,Aam Aadmi Party,Assam,2,0
4,INDIA bloc,Aam Aadmi Party,Haryana,1,0
...,...,...,...,...,...
254,Other,United Democratic Party,Meghalaya,1,<NA>
255,Other,Uttarakhand Kranti Dal,Uttarakhand,3,<NA>
256,Other,Voice of the People Party,Meghalaya,1,<NA>
257,Other,YSR Congress Party,Andhra Pradesh,25,<NA>


In [10]:
# Every party, ranked: seats contested vs seats won, and where
summary = (results.groupby(["alliance", "party"])
           .agg(states=("state", "nunique"),
                contested=("contested", "sum"),
                won=("won", lambda s: s.sum(min_count=1)))   # keep <NA> as <NA>, not 0
           .sort_values("won", ascending=False))
summary.head(15)

states  \
alliance   party                                                      
NDA        Bharatiya Janata Party                                33   
INDIA bloc Indian National Congress                              36   
           Samajwadi Party                                        4   
           All India Trinamool Congress                           4   
           Dravida Munnetra Kazhagam                              1   
NDA        Telugu Desam Party                                     1   
           Janata Dal (United)                                    1   
INDIA bloc Shiv Sena (Uddhav Balasaheb Thackeray)                 1   
           Nationalist Congress Party (Sharadchandra Pawar)       3   
NDA        Shiv Sena                                              1   
           Lok Janshakti Party (Ram Vilas)                        1   
INDIA bloc Rashtriya Janata Dal                                   2   
           Communist Party of India (Marxist)                    15   
           Indian Union Muslim League                             2   
           Aam Aadmi Party                                        5   

                                                             contested  won  
alliance   party                                                             
NDA        Bharatiya Janata Party                                  441  240  
INDIA bloc Indian National Congress                                328   99  
           Samajwadi Party                                          71   37  
           All India Trinamool Congress                             48   29  
           Dravida Munnetra Kazhagam                                21   21  
NDA        Telugu Desam Party                                       17   16  
           Janata Dal (United)                                      16   12  
INDIA bloc Shiv Sena (Uddhav Balasaheb Thackeray)                   21    9  
           Nationalist Congress Party (Sharadchandra Pawar)         12    8  
NDA        Shiv Sena                                                15    7  
           Lok Janshakti Party (Ram Vilas)                           5    5  
INDIA bloc Rashtriya Janata Dal                                     24    4  
           Communist Party of India (Marxist)                       52    4  
           Indian Union Muslim League                                3    3  
           Aam Aadmi Party                                          22    3

In [11]:
# One state, party by party — change the name and rerun
results[results["state"].eq("Tamil Nadu")].sort_values("won", ascending=False).head(10)

,alliance,party,state,contested,won
104,INDIA bloc,Dravida Munnetra Kazhagam,Tamil Nadu,21,21
75,INDIA bloc,Indian National Congress,Tamil Nadu,9,9
97,INDIA bloc,Communist Party of India (Marxist),Tamil Nadu,2,2
111,INDIA bloc,Communist Party of India,Tamil Nadu,2,2
127,INDIA bloc,Viduthalai Chiruthaigal Katchi,Tamil Nadu,2,2
124,INDIA bloc,Indian Union Muslim League,Tamil Nadu,1,1
105,INDIA bloc,Kongunadu Makkal Desia Katchi,Tamil Nadu,1,1
132,INDIA bloc,Marumalarchi Dravida Munnetra Kazhagam,Tamil Nadu,1,1
58,NDA,Independent,Tamil Nadu,1,0
7,NDA,Bharatiya Janata Party,Tamil Nadu,23,0


In [12]:
results.to_csv("election_2024_party_state.csv", index=False)
print(f"saved {len(results)} rows to election_2024_party_state.csv")

saved 259 rows to election_2024_party_state.csv


## What to take away

- **`read_html` turns rung 3 into two lines** — when the site is static HTML with real tables.
  Everything else was data cleaning, and that ratio (one line of fetching, forty of cleaning) is
  the normal shape of scraping work.
- **Select tables by signature, never by index.** The page will be edited; your column names are
  more stable than your table positions.
- **Merged cells become repeated values** in pandas — that is where the phantom double-counting
  came from, and why the `Total` rows and the totals columns had to go.
- **Keep the source's own totals as a free audit** rather than as data. When your bottom-up sum
  matches their top-down figure, both of you are probably right — and note the audit only
  balanced once we noticed one alliance was split across two tables.
- **Never turn "the source doesn't say" into a zero.** The no-`won`-column table held real
  winners; `<NA>` keeps the gap visible, a fabricated 0 would have hidden it forever.
- Wikipedia allows this kind of read and asks only for an honest User-Agent — for heavier use
  there is a proper [API](https://www.mediawiki.org/wiki/API:Main_page) and full database dumps,
  the rung-1 options. One page, once, parsed from memory: rung 3 is fine.